# 5.4. Numerical Stability and Initialization
D2L의 Numerical Stability and Initialization장을 PyTorch 기준으로 정리함.

1. Vanishing Gradient
2. Exploding Gradient
3. Symmetry 문제
4. Weight Initialization
5. Xavier Initialization

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 깊은 신경망과 Gradient

예를 들어서 이런 신경망이 있다고 하자.

```text
X
↓
Layer 1
↓
Layer 2
↓
Layer 3
↓
Output
↓
Loss
```

순전파에서는 앞에서 뒤로 계산한다.

    X -> Layer1 -> Layer2 -> Layer3 -> output -> Loss

역전파에서는 반대로 gradient가 전달된다.

    Loss -> Layer3 gradient -> Layer2 gradient -> Layer1 gradient

이때 미분값들이 계속 곱해진다.

지난 장의 Chain Rule이 여기서 나온다.

$$
x \rightarrow h_1 \rightarrow h_2 \rightarrow y \rightarrow L
$$

이면

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\times\frac{\partial y}{\partial h_2}\times\frac{\partial h_2}{\partial h_1}\times\frac{\partial h_1}{\partial x}
$$

이다. D2L에서도 같은 네트워크의 gradient가 여러 층의 미분 행렬을 연속해서 곱한 형태가 되기 때문에 결과가 매우 커지거나 매우 작아질 수 있다고 설명한다.

예를 들어
```text
각 층의 미분값 = 0.5

Layer가 3개라면

0.5 x 0.5 x 0.5 = 0.125

Layer 10개라면

0.5^10 = 0.000977

반대로 각 층에서 2가 곱해진다면

2^10 = 1024 가 된다.
```

    작은 값을 계속 곱하면 0에 가까워지고
    큰 값을 계속 곱하면 엄청 커진다.


## 2. Vanishing Gradient

